# 07 -- Scikit-learn : introduction

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- comprendre l'API uniforme `fit` / `predict` / `transform` ;
- charger et separer un jeu de donnees en train/test ;
- entrainer des modeles de regression et de classification ;
- evaluer les performances avec les metriques appropriees ;
- utiliser des pipelines et des transformateurs ;
- encoder des variables categorielles et standardiser ;
- utiliser la validation croisee et la recherche d'hyperparametres ;
- sauvegarder et charger un modele.

## Prerequis -- ce que vous devez maitriser

- NumPy (arrays, indexation) et Pandas (DataFrame) ;
- les bases statistiques : moyenne, variance, correlation ;
- les notions supervise vs non supervise (aide-memoire en section 1).

Ce que ce notebook **ne couvrira pas** :

- le deep learning (PyTorch, Keras) ;
- les modeles avances (XGBoost, LightGBM) ;
- les reseaux de neurones.

## Plan

1. Installation et philosophie
2. Charger un dataset
3. Train/test split
4. Regression lineaire
5. Evaluation de regression
6. Classification
7. Evaluation de classification
8. Preprocessing et encodage
9. Pipelines
10. Validation croisee
11. GridSearchCV
12. Clustering (non supervise)
13. Reduction de dimensions (PCA)
14. Sauvegarder un modele
15. Synthese
16. Exercices

---

## 1. Installation et philosophie

In [ ]:
# pip install scikit-learn
import sklearn
sklearn.__version__

### API uniforme

Tout estimateur a la meme interface :

- `fit(X, y)` : entrainer le modele
- `predict(X)` : predire des valeurs
- `transform(X)` : transformer les donnees (preprocessing)
- `score(X, y)` : evaluer le modele

`X` est toujours 2D (n_samples, n_features) et `y` est 1D (n_samples,).

---

## 2. Charger un dataset

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()
type(iris)

In [ ]:
iris.data.shape  # (n_samples, n_features)

In [ ]:
iris.target[:10]

In [ ]:
iris.feature_names

In [ ]:
iris.target_names

### Datasets natifs

- `load_iris`, `load_digits`, `load_wine`, `load_diabetes`
- `fetch_california_housing`, `fetch_openml('mnist_784')`

---

## 3. Train/test split

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)
print('Train:', X_train.shape, 'Test:', X_test.shape)

`random_state` rend le split reproductible. `stratify=y` preserve les proportions de classes.

In [ ]:
# Version stratifiee
X_tr, X_te, y_tr, y_te = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)
X_tr.shape

---

## 4. Regression lineaire

In [ ]:
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
X, y = housing.data, housing.target
X.shape, y.shape

In [ ]:
from sklearn.linear_model import LinearRegression
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
model = LinearRegression()
model.fit(X_tr, y_tr)

In [ ]:
model.coef_

In [ ]:
model.intercept_

In [ ]:
predictions = model.predict(X_te)
predictions[:5]

---

## 5. Evaluation de regression

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
mse = mean_squared_error(y_te, predictions)
print('MSE :', round(mse, 3))

In [ ]:
print('RMSE :', round(mse**0.5, 3))

In [ ]:
print('R2   :', round(r2_score(y_te, predictions), 3))

In [ ]:
print('MAE  :', round(mean_absolute_error(y_te, predictions), 3))

---

## 6. Classification

In [ ]:
from sklearn.linear_model import LogisticRegression
X_tr, X_te, y_tr, y_te = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_tr, y_tr)

In [ ]:
clf.predict(X_te[:5])

In [ ]:
# Probabilites
clf.predict_proba(X_te[:3]).round(3)

### Autres classifieurs courants

In [ ]:
from sklearn.tree import DecisionTreeClassifier
tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(X_tr, y_tr)
tree.score(X_te, y_te)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=0)
rf.fit(X_tr, y_tr)
rf.score(X_te, y_te)

In [ ]:
from sklearn.svm import SVC
svc = SVC(kernel='rbf', gamma='scale')
svc.fit(X_tr, y_tr)
svc.score(X_te, y_te)

---

## 7. Evaluation de classification

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
y_pred = clf.predict(X_te)
accuracy_score(y_te, y_pred)

In [ ]:
confusion_matrix(y_te, y_pred)

In [ ]:
print(classification_report(y_te, y_pred, target_names=iris.target_names))

### Vocabulaire

- **Precision** = TP / (TP + FP) : parmi les positifs predits, combien sont corrects ?
- **Recall**    = TP / (TP + FN) : parmi les positifs reels, combien sont trouves ?
- **F1**        = moyenne harmonique precision/recall
- **Support**   = nombre d'exemples dans la classe

---

## 8. Preprocessing et encodage

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
X = np.array([[1.0, 200.0], [2.0, 300.0], [3.0, 400.0]])
scaler = StandardScaler()
scaler.fit_transform(X)

### Encodage one-hot

In [ ]:
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder(sparse_output=False)
enc.fit_transform([['rouge'], ['vert'], ['bleu'], ['rouge']])

### Encodage ordinal (labels)

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit_transform(['chat', 'chien', 'chat', 'oiseau'])

### Imputation des valeurs manquantes

In [ ]:
from sklearn.impute import SimpleImputer
imp = SimpleImputer(strategy='mean')
imp.fit_transform([[1.0, 2.0], [float('nan'), 3.0], [7.0, 6.0]])

---

## 9. Pipelines

Un **Pipeline** chaine plusieurs etapes (preprocessing + modele) en un seul objet.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000)),
])
pipe.fit(X_tr, y_tr)
pipe.score(X_te, y_te)

### ColumnTransformer : preprocessing different par colonne

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer([
    ('num', StandardScaler(), [0, 1]),
    ('cat', OneHotEncoder(), [2]),
])
print(type(ct).__name__, 'pret a etre utilise dans un Pipeline')

---

## 10. Validation croisee

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(clf, iris.data, iris.target, cv=5)
print('Scores :', scores.round(3))
print('Moyenne :', scores.mean().round(3), '+/-', scores.std().round(3))

In [ ]:
# K-fold stratifie
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cross_val_score(clf, iris.data, iris.target, cv=cv).mean()

---

## 11. GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.01, 0.1],
    'kernel': ['rbf', 'linear'],
}
grid = GridSearchCV(SVC(), param_grid, cv=3, n_jobs=-1)
grid.fit(iris.data, iris.target)

In [ ]:
grid.best_params_

In [ ]:
grid.best_score_.round(3)

---

## 12. Clustering (non supervise)

In [ ]:
from sklearn.cluster import KMeans
X = iris.data
km = KMeans(n_clusters=3, n_init=10, random_state=0)
km.fit(X)

In [ ]:
km.labels_[:20]

In [ ]:
km.cluster_centers_

### DBSCAN : clustering base densite

In [ ]:
from sklearn.cluster import DBSCAN
db = DBSCAN(eps=0.5, min_samples=5)
db.fit(X)
import numpy as np
np.unique(db.labels_, return_counts=True)

---

## 13. Reduction de dimensions (PCA)

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_2d = pca.fit_transform(iris.data)
X_2d.shape

In [ ]:
pca.explained_variance_ratio_

In [ ]:
# Variance cumulee
pca.explained_variance_ratio_.cumsum()

---

## 14. Sauvegarder un modele

In [ ]:
import joblib
import tempfile, pathlib
path = pathlib.Path(tempfile.gettempdir()) / 'model.joblib'
joblib.dump(clf, path)
print('Taille :', path.stat().st_size, 'octets')

In [ ]:
# Charger
clf_loaded = joblib.load(path)
clf_loaded.score(X_te, y_te)

Pour la production, preferer **ONNX** qui est independant de la version scikit-learn.

---

## Synthese

| Tache | Classe scikit-learn |
|---|---|
| Regression | `LinearRegression`, `Ridge`, `Lasso` |
| Classification | `LogisticRegression`, `DecisionTreeClassifier`, `RandomForestClassifier`, `SVC` |
| Clustering | `KMeans`, `DBSCAN`, `AgglomerativeClustering` |
| Preprocessing | `StandardScaler`, `OneHotEncoder`, `SimpleImputer` |
| Chaine | `Pipeline`, `ColumnTransformer` |
| Evaluation | `cross_val_score`, `GridSearchCV` |
| Metriques regression | `mean_squared_error`, `r2_score` |
| Metriques classification | `accuracy_score`, `classification_report` |

### Regles a retenir

- **Fit sur train seulement**, jamais sur test.
- **Pipeline = reproductibilite** : evite le data leakage en CV.
- **Stratify pour les classes desequilibrees**.
- **Always set `random_state`** pour la reproductibilite.
- Choisir la metrique selon le probleme : **F1 pour desequilibre**, **RMSE pour regression**.

---

## Exercices

### Exercice 1 (facile) -- Classifier les iris

Chargez le dataset iris, faites un split 70/30, entrainez une `LogisticRegression` et affichez l'accuracy.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# TODO

<details>
<summary>Voir la correction</summary>

```python
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

iris = load_iris()
X_tr, X_te, y_tr, y_te = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=0
)
clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
print('Accuracy :', accuracy_score(y_te, clf.predict(X_te)))
```

</details>

### Exercice 2 (facile) -- Regression sur diabetes

Entrainez une regression lineaire sur `load_diabetes`, affichez le R2 sur le test.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

d = load_diabetes()
X_tr, X_te, y_tr, y_te = train_test_split(d.data, d.target, test_size=0.2, random_state=0)
model = LinearRegression().fit(X_tr, y_tr)
print('R2 :', round(r2_score(y_te, model.predict(X_te)), 3))
```

</details>

### Exercice 3 (moyen) -- Pipeline avec standardisation

Creez un Pipeline avec `StandardScaler` + `LogisticRegression`, entrainez-le sur iris et evaluez-le avec `cross_val_score(cv=5)`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_iris

iris = load_iris()
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000)),
])
scores = cross_val_score(pipe, iris.data, iris.target, cv=5)
print('Scores :', scores.round(3))
print('Moyenne :', scores.mean().round(3))
```

</details>

### Exercice 4 (moyen) -- GridSearch sur RandomForest

Utilisez `GridSearchCV` pour trouver les meilleurs hyperparametres d'un `RandomForestClassifier` sur iris (`n_estimators` dans [10, 50, 100], `max_depth` dans [2, 4, None]).

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import load_iris

iris = load_iris()
grid = GridSearchCV(
    RandomForestClassifier(random_state=0),
    {'n_estimators': [10, 50, 100], 'max_depth': [2, 4, None]},
    cv=5,
)
grid.fit(iris.data, iris.target)
print('Best :', grid.best_params_, '-> score', round(grid.best_score_, 3))
```

</details>

### Exercice 5 (difficile) -- Clustering et PCA

Sur le dataset iris :
1. Appliquez PCA en 2D ;
2. Appliquez KMeans (3 clusters) sur les donnees originales ;
3. Affichez le score **adjusted rand index** en comparant les clusters avec les vraies classes.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.metrics import adjusted_rand_score

iris = load_iris()
X_2d = PCA(n_components=2).fit_transform(iris.data)
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(iris.data)
print('ARI :', round(adjusted_rand_score(iris.target, km.labels_), 3))
print('PCA shape :', X_2d.shape)
```

</details>

---

## Ressources externes

- [Documentation scikit-learn](https://scikit-learn.org/stable/)
- [User guide](https://scikit-learn.org/stable/user_guide.html)
- [Examples gallery](https://scikit-learn.org/stable/auto_examples/index.html)
- [Choosing the right estimator](https://scikit-learn.org/stable/machine_learning_map.html)